In [ ]:
## importing the Libraries

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
import sys
print(sys.executable)

import optuna
print(optuna.__version__)
from imblearn.over_sampling import SMOTE

C:\Users\Venky\anaconda3\Anakkonda\python.exe
4.9.0


## load the dataset

In [3]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")
fake["label"] = 0
true["label"] = 1
df = pd.concat([fake, true], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.columns.tolist())

['title', 'text', 'subject', 'date', 'label']


## Basic cleaning / feature engineering

In [4]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["subject"] = df["subject"].fillna("unknown")

In [5]:
df["full_text"] = df["title"] + " " + df["text"]

In [6]:
df["text_length"] = df["full_text"].apply(len)

In [7]:
df = df.drop(columns=["date", "title", "text"])

## EDA

In [8]:
print("Fake Dataset Shape :", fake.shape)
print("True Dataset Shape :", true.shape)
print("Merged Dataset Shape :", df.shape)

Fake Dataset Shape : (23481, 5)
True Dataset Shape : (21417, 5)
Merged Dataset Shape : (44898, 4)


In [9]:
df.head()

,subject,label,full_text,text_length
0,US_News,0,Ben Stein Calls Out 9th Circuit Court: Committ...,1118
1,politicsNews,1,Trump drops Steve Bannon from National Securit...,4876
2,politicsNews,1,Puerto Rico expects U.S. to lift Jones Act shi...,1913
3,News,0,OOPS: Trump Just Accidentally Confirmed He Le...,1334
4,politicsNews,1,Donald Trump heads for Scotland to reopen a go...,3193


In [10]:
df.to_csv("fake_news_dataset.csv", index=False)

In [11]:
print(df["label"].value_counts())

label
0    23481
1    21417
Name: count, dtype: int64


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   subject      44898 non-null  str  
 1   label        44898 non-null  int64
 2   full_text    44898 non-null  str  
 3   text_length  44898 non-null  int64
dtypes: int64(2), str(2)
memory usage: 111.4 MB


In [13]:
df.head()

,subject,label,full_text,text_length
0,US_News,0,Ben Stein Calls Out 9th Circuit Court: Committ...,1118
1,politicsNews,1,Trump drops Steve Bannon from National Securit...,4876
2,politicsNews,1,Puerto Rico expects U.S. to lift Jones Act shi...,1913
3,News,0,OOPS: Trump Just Accidentally Confirmed He Le...,1334
4,politicsNews,1,Donald Trump heads for Scotland to reopen a go...,3193


In [14]:
df.tail()

,subject,label,full_text,text_length
44893,politics,0,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,76
44894,worldnews,1,PM May seeks to ease Japan's Brexit fears duri...,4409
44895,worldnews,1,Merkel: Difficult German coalition talks can r...,492
44896,News,0,Trump Stole An Idea From North Korean Propaga...,5159
44897,politics,0,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...,1564


In [15]:
df.shape

(44898, 4)

In [16]:
df.isnull().sum()

subject        0
label          0
full_text      0
text_length    0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(213)

In [18]:
df = df.drop_duplicates()

In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
df.describe()

,label,text_length
count,44685.000000,44685.000000
mean,0.474611,2548.140002
std,0.499361,2174.536106
min,0.000000,31.000000
25%,0.000000,1314.000000
50%,0.000000,2268.000000
75%,1.000000,3186.000000
max,1.000000,51893.000000


## DATA SPLITING

In [21]:
X=df.drop("label",axis=1)
y=df['label']

In [22]:
X_train_fully,X_test,y_train_fully,y_test=train_test_split(X,y,test_size=0.2,random_state=2,stratify=y)

In [23]:
X_train,X_val,y_train,y_val=train_test_split(X_train_fully,y_train_fully,test_size=0.2,random_state=2,stratify=y_train_fully)

In [24]:
 print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (28598, 3) Val: (7150, 3) Test: (8937, 3)


## preprocessor

In [25]:
cat_cols = X_train.select_dtypes(include='object').columns
num_cols = X_train.select_dtypes(include='number').columns

C:\Users\Venky\AppData\Local\Temp\ipykernel_20184\1518593067.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns


In [26]:
print(num_cols)
print(cat_cols)

Index(['text_length'], dtype='str')
Index(['subject', 'full_text'], dtype='str')


In [27]:
text_col = "full_text"
cat_cols = ["subject"]
num_cols = ["text_length"]

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=5000, stop_words="english"), text_col),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ]
)

In [29]:
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

X_train_proc = X_train_proc.toarray()
X_val_proc = X_val_proc.toarray()
X_test_proc = X_test_proc.toarray()

input_dim = X_train_proc.shape[1]

## ANN MODEL

In [30]:
model=Sequential()

In [31]:
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

## COMPILE

In [32]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [33]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 256)                 │       1,282,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,299,073 (4.96 MB)

 Trainable params: 1,299,073 (4.96 MB)

 Non-trainable params: 0 (0.00 B)

## TRAINING

In [34]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

In [35]:
history = model.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=20,
    batch_size=120,
    callbacks=[early_stop],
)

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.9942 - loss: 0.0576 - val_accuracy: 0.9999 - val_loss: 3.2924e-04
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 1.0000 - loss: 1.7451e-04 - val_accuracy: 0.9999 - val_loss: 2.2496e-04
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 1.0000 - loss: 6.3395e-05 - val_accuracy: 0.9999 - val_loss: 1.8966e-04
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 1.0000 - loss: 3.7172e-05 - val_accuracy: 0.9999 - val_loss: 1.5070e-04
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 1.0000 - loss: 2.1590e-05 - val_accuracy: 0.9999 - val_loss: 1.2888e-04
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 1.0000 - loss: 1.6441e-05 - val_accuracy: 1.0000 - val_loss: 9.1331e-05
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 1.0000 - loss: 1.2371e-05 - val_accuracy: 0.9999 - val_loss: 1.7824e-04
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 5s 2

In [36]:
test_loss, test_acc = model.evaluate(X_test_proc, y_test)
print(f"Test Accuracy: {test_acc:.4f}")
 
y_pred = (model.predict(X_test_proc) > 0.5).astype(int)

280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 6.3243e-06 
Test Accuracy: 1.0000
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [37]:
print(confusion_matrix(y_test, y_pred))

[[4695    0]
 [   0 4242]]


In [38]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4695
           1       1.00      1.00      1.00      4242

    accuracy                           1.00      8937
   macro avg       1.00      1.00      1.00      8937
weighted avg       1.00      1.00      1.00      8937



## apply SMOTE

In [39]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)

print("\nClass balance after SMOTE:")
print(pd.Series(y_train_res).value_counts())
print(X_train_res.shape, y_train_res.shape)


Class balance after SMOTE:
label
1    15025
0    15025
Name: count, dtype: int64
(30050, 5009) (30050,)


## Hyper parameter tuning

In [40]:
def build_model(trial, lr_rate, n_layers, optimizer_name, activation, hidden_units):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
 
    for i in range(n_layers):
        model.add(layers.Dense(hidden_units[i], activation=activation.lower()))
        model.add(layers.BatchNormalization())
        dropout_rate = trial.suggest_float(f"dropout_{i}", 0.1, 0.5, step=0.1)
        model.add(layers.Dropout(dropout_rate))
 
    model.add(layers.Dense(1, activation="sigmoid"))
 
    if optimizer_name == "Adam":
        optimizer = keras.optimizers.Adam(learning_rate=lr_rate)
    elif optimizer_name == "RMSprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=lr_rate)
    else:  # SGD
        optimizer = keras.optimizers.SGD(learning_rate=lr_rate)
 
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model
 
 
def objective(trial):
    # --- exact parameters from the handwritten notes ---
    lr_rate = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_int("n_layers", 1, 4)
    optimizer_name = trial.suggest_categorical("opt", ["Adam", "RMSprop", "SGD"])
    activation = trial.suggest_categorical("activation", ["Tanh", "Relu"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512])
 
    # units per layer (one extra tunable param per layer)
    hidden_units = [
        trial.suggest_categorical(f"units_{i}", [16, 32, 64, 128, 256])
        for i in range(n_layers)
    ]
 
    model = build_model(trial, lr_rate, n_layers, optimizer_name, activation, hidden_units)
 
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
 
    history = model.fit(
        X_train_res, y_train_res,
        validation_data=(X_val_proc, y_val),
        epochs=20,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0,
    )
 
    val_accuracy = max(history.history["val_accuracy"])
    return val_accuracy

## optima study

In [41]:
study = optuna.create_study(direction="maximize", study_name="fake_news_ann_optuna")
study.optimize(objective, n_trials=30, show_progress_bar=True)
 
print("\nBest trial:")
print(f"  Value (val_accuracy): {study.best_trial.value:.4f}")
print("  Params:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")
 
joblib.dump(study, "optuna_study.pkl")

[I 2026-07-03 11:55:37,444] A new study created in memory with name: fake_news_ann_optuna


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-07-03 11:56:00,742] Trial 0 finished with value: 1.0 and parameters: {'lr': 0.0005296982144715005, 'n_layers': 3, 'opt': 'RMSprop', 'activation': 'Tanh', 'batch_size': 128, 'units_0': 16, 'units_1': 128, 'units_2': 128, 'dropout_0': 0.5, 'dropout_1': 0.4, 'dropout_2': 0.4}. Best is trial 0 with value: 1.0.
[I 2026-07-03 11:56:35,315] Trial 1 finished with value: 0.9998601675033569 and parameters: {'lr': 0.003842082435650347, 'n_layers': 3, 'opt': 'RMSprop', 'activation': 'Relu', 'batch_size': 128, 'units_0': 256, 'units_1': 128, 'units_2': 256, 'dropout_0': 0.4, 'dropout_1': 0.1, 'dropout_2': 0.1}. Best is trial 0 with value: 1.0.
[I 2026-07-03 11:57:24,070] Trial 2 finished with value: 1.0 and parameters: {'lr': 0.00032630651377927016, 'n_layers': 2, 'opt': 'Adam', 'activation': 'Relu', 'batch_size': 64, 'units_0': 128, 'units_1': 64, 'dropout_0': 0.4, 'dropout_1': 0.1}. Best is trial 0 with value: 1.0.
[I 2026-07-03 11:57:47,981] Trial 3 finished with value: 1.0 and parameter

['optuna_study.pkl']

## evalution

In [42]:
best_params = study.best_trial.params
best_n_layers = best_params["n_layers"]
best_hidden_units = [best_params[f"units_{i}"] for i in range(best_n_layers)]
 
best_model = build_model(
    study.best_trial,
    lr_rate=best_params["lr"],
    n_layers=best_n_layers,
    optimizer_name=best_params["opt"],
    activation=best_params["activation"],
    hidden_units=best_hidden_units,
)
 
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
 
best_model.fit(
    X_train_res, y_train_res,
    validation_data=(X_val_proc, y_val),
    epochs=50,
    batch_size=best_params["batch_size"],
    callbacks=[early_stop],
)
 
test_loss, test_acc = best_model.evaluate(X_test_proc, y_test)
print(f"\nTest Accuracy (Optuna-tuned ANN): {test_acc:.4f}")
 
y_pred = (best_model.predict(X_test_proc) > 0.5).astype(int)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Epoch 1/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.9741 - loss: 0.0612 - val_accuracy: 0.9999 - val_loss: 0.0079
Epoch 2/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9997 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 9.4868e-07
Epoch 3/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9998 - loss: 7.4530e-04 - val_accuracy: 1.0000 - val_loss: 2.1243e-05
Epoch 4/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9999 - loss: 2.1722e-04 - val_accuracy: 1.0000 - val_loss: 6.3964e-09
Epoch 5/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9999 - loss: 2.4574e-04 - val_accuracy: 1.0000 - val_loss: 6.1549e-09
Epoch 6/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9999 - loss: 2.4547e-04 - val_accuracy: 1.0000 - val_loss: 1.2414e-09
Epoch 7/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9999 - loss: 2.3885e-04 - val_accuracy: 1.0000 - val_loss: 2.6791e-09
Epoch 8/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - acc

## deployment

In [43]:
best_model.save("fake_news_model.keras")
print("\nSaved: fake_news_model.keras, preprocessor.pkl, feature_columns.pkl, optuna_study.pkl")


Saved: fake_news_model.keras, preprocessor.pkl, feature_columns.pkl, optuna_study.pkl
